In [ ]:
import os
import json
from dotenv import load_dotenv
from typing_extensions import TypedDict, Annotated
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from tools.redmine import get_issues, get_members, get_versions, get_projects

**AgentState:** State tracks the **plan**, current **step**, and **messages**

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    plan: list[str]
    index: int
    observations: list[str]

In [ ]:
load_dotenv()

**Model Declaration:**

In [ ]:
llm = ChatOpenAI(
    model="openrouter/auto",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)

**Tool Calling:**

In [ ]:
tools = [get_projects, get_issues, get_members, get_versions]
TOOL_REGISTRY = {tool.name: tool for tool in tools}

**Planner**, generates the plan

In [ ]:
PLANNER_PROMPT = """You are a planning assistant for a Redmine project management chatbot.
You have access to ONLY these tools:
- get_projects : retrieves all available projects
- get_issues   : retrieves issues with filters (project_id, status_id, priority_id, due_before, assigned_to_id)
- get_members  : retrieves members of a project
- get_versions : retrieves sprints/versions of a project
 
Break the user request into an ordered list of tool calls needed to answer it.
Each step must specify exactly which tool to call and with which parameters.
 
Respond ONLY with a valid JSON array of objects, like this:
[
  {"tool": "get_projects", "args": {}},
  {"tool": "get_issues", "args": {"project_id": "e-commerce-platform", "status_id": "open"}},
  {"tool": "get_members", "args": {"project_id": "e-commerce-platform"}}
]
 
Rules:
- Only use tool names from the list above
- Always include "tool" and "args" keys in each step
- If you don't know the project_id yet, add get_projects as the first step
- Never add steps like "parse response" or "summarize" — only tool calls
"""

In [ ]:
def planner(state: AgentState) -> AgentState:
    user_message = state["messages"][-1]
    
    response = llm.invoke([
        ("system", PLANNER_PROMPT),
        ("human", f"User request: {user_message.content}")
    ])
    
    # Parse the JSON plan robustly
    try:
        raw = response.content.strip()
        # Strip markdown code blocks if present
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        plan = json.loads(raw.strip())
    except json.JSONDecodeError:
        # Fallback — if parsing fails, default to getting projects
        print(f"[Planner] Failed to parse plan, using fallback. Raw: {response.content}")
        plan = [{"tool": "get_projects", "args": {}}]
    
    print(f"\n📋 PLAN ({len(plan)} steps):")
    for i, step in enumerate(plan):
        print(f"   Step {i+1}: {step['tool']}({step.get('args', {})})")
    print()
    
    return {
        "plan": plan,
        "index": 0,
        "observations": []
    }

**Executor** processes one step, calling tools if needed

In [ ]:
def executor(state: AgentState) -> AgentState:
    plan = state["plan"]
    index = state["index"]
    
    if index >= len(plan):
        return {"index": index}
    
    step = plan[index]
    tool_name = step.get("tool", "")
    tool_args = step.get("args", {})
    
    print(f"STEP {index + 1}/{len(plan)} — Executing: {tool_name}({tool_args})")
    
    if tool_name in TOOL_REGISTRY:
        try:
            tool_func = TOOL_REGISTRY[tool_name]
            result = tool_func.invoke(tool_args)
            print(f"Result: {json.dumps(result, ensure_ascii=False)[:200]}...")
        except Exception as e:
            result = {"error": str(e)}
            print(f"Tool error: {e}")
    else:
        result = {"error": f"Unknown tool: {tool_name}"}
        print(f"Unknown tool: {tool_name}")
    
    # Accumulate observations — preserve all previous ones
    previous_observations = state.get("observations", [])
    new_observation = {
        "step": index + 1,
        "tool": tool_name,
        "args": tool_args,
        "result": result
    }
    
    return {
        "observations": previous_observations + [new_observation],
        "index": index + 1
    }

In [ ]:
def should_continue(state: AgentState) -> str:
    if state["index"] < len(state["plan"]):
        return "executor"
    return "aggregator"

**Aggregation Agent** collects the executor's output into one cohesive response

In [ ]:
AGGREGATOR_PROMPT = """You are a project management assistant.
You have executed a series of Redmine API calls to answer the user's question.
Below are all the results collected from those calls.
Write a clear, structured, and complete answer in French based strictly on this data.
Do not invent any data — only use what is provided in the observations."""

In [ ]:
def aggregator(state: AgentState) -> dict:
    observations = state.get("observations", [])
    user_question = state["messages"][0].content
    
    print(f"\nAGGREGATING {len(observations)} observations...")
    
    # Format all observations for the LLM
    formatted_observations = ""
    for obs in observations:
        formatted_observations += f"\nStep {obs['step']} — {obs['tool']}({obs['args']}):\n"
        formatted_observations += json.dumps(obs["result"], ensure_ascii=False, indent=2)
        formatted_observations += "\n"
    
    response = llm.invoke([
        ("system", AGGREGATOR_PROMPT),
        ("human", f"""User question: {user_question}
 
Data collected from Redmine:
{formatted_observations}
 
Now write the final answer for the user.""")
    ])
    
    print("\n─" * 25)
    print("FINAL ANSWER:")
    print(response.content)
    print("─" * 25)
    
    return {
        "messages": [response],
        "observations": observations
    }

**Build the Graph**

In [ ]:
builder = StateGraph(AgentState)
 
builder.add_node("planner", planner)
builder.add_node("executor", executor)
builder.add_node("aggregator", aggregator)
 
builder.set_entry_point("planner")
builder.add_edge("planner", "executor")
builder.add_conditional_edges(
    "executor",
    should_continue,
    {
        "executor": "executor",
        "aggregator": "aggregator"
    }
)
builder.add_edge("aggregator", END)
 
app = builder.compile()

**Interactions**

In [ ]:
result = app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Give me a list of all open issues in the e-commerce-platform project and the members assigned to them."
            }
        ],
        "observations": []
    }
)

In [ ]:
result

In [ ]:
print(result["messages"][-1].content)

**Mermaid graph visualisation:**

In [ ]:
print(app.get_graph().draw_mermaid())